# 86장 · 서비스 문제 정의와 최소 시연 범위

최소 시연 범위는 한 사용자의 어떤 질문에 무슨 근거로 답할지 정한 약속입니다. 첫 시연에서 제외할 일도 함께 적습니다.

최종 프로젝트의 예시는 가상 장비실 규정 조회 서비스입니다. 장비 A의 점검 질문에 답하고 해당 규정의 절을 제시하는 경로를 첫 시연 범위로 정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 배포 판단에서 첫 시연 조건 고르기

팀이 만들 서비스는 장비별 점검 절차를 안내합니다. 85장에서 기대와 관찰이 달랐던 사례를 읽고 첫 시연에서 같은 문제가 생기면 무엇을 확인할지 적습니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `service-lab/evaluation/release.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter86-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "service-lab/evaluation/release.json"
sample_file = here / "chapter86-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['case_id'])

교재 예시 chapter86-start.json
S01


두 번째 줄은 읽은 파일의 `case_id` 값입니다.

## 사용자 질문과 기대 답변

문제 정의 예시　교대 담당자는 장비 A의 점검 전 확인사항을 찾으려고 규정 여러 쪽을 뒤집니다. 질문을 입력하면 해당 문장과 절·개정일을 함께 보여 줍니다. 담당자는 그 자리에서 원문을 열어 자신이 찾던 규정인지 대조합니다. 이 사용 장면은 기획 가정이며 현장 인터뷰 결과는 아직 없습니다.

점검 승인까지 자동화하려면 실제 담당 권한과 승인 이력이 필요합니다. 이 팀은 이미 준비한 가상 규정으로 검증할 수 있는 ‘근거 찾기’를 첫 시연으로 골랐습니다. 기획자는 질문을 잡고 자료 담당자는 원문 위치를 확인합니다. 서로 다른 사람이 똑같은 절을 찾으면 다음 장의 시나리오로 넘길 수 있습니다.


### 셀 01 · 최소 시연의 입력과 기대 답변

팀이 가진 자료에 실제로 답의 근거가 있는지 확인해 보세요.

선정 메모에 실제 입력을 한 줄 남기세요. ‘장비 A 점검 전에 무엇을 확인하나요?’에 답할 근거는 규정 A 2.1절의 ‘점검 전 전원 표시등을 확인합니다.’입니다. 다음 팀원에게 질문과 원문 위치를 함께 전달합니다.


In [1]:
rule = {'id': 'RULE-A',
 'equipment': '장비 A',
 'text': '점검 전 전원 표시등을 확인합니다.',
 'source': '가상 장비실 규정 A',
 'page': 1,
 'section': '2.1',
 'revised_on': '2026-11-01',
 'version': 'rules-v1',
 'groups': ['public', 'admin']}
print(rule["text"])
print(rule["source"], rule["section"], rule["revised_on"])

점검 전 전원 표시등을 확인합니다.
가상 장비실 규정 A 2.1 2026-11-01


표시등을 확인하라는 문장 아래에 규정 A, 2.1, 개정일이 보입니다. 이 자료로 “어디에 쓰여 있나요?”에 답하는 시연을 만들 수 있습니다. 점검을 누가 승인했는지는 이 레코드에 없습니다.

가상 규정에서 장비 A와 B의 점검 문구는 다릅니다. 팀의 첫 시연을 설계할 때도 ‘질문에 답한다’에서 멈추지 않고, 어떤 대상의 질문인지와 확인할 규정의 절 번호를 정해 둡니다. 그래야 다른 장비의 문장을 가져온 답을 실패로 판정할 수 있습니다.


### 규정 문장과 원문 위치를 한 쌍으로 읽기

첫 줄은 교대 담당자가 수행할 행동입니다. 둘째 줄의 2.1은 원문을 다시 펼칠 절 번호이고 2026-11-01은 그 규정의 개정일입니다. 담당자가 다른 판본을 열었다면 절 번호가 같아도 같은 지침인지 개정일부터 대조해야 합니다.


## 제외 범위와 주제 선정 근거

채워진 메모　프로젝트 PROJECT-EQUIPMENT-01 · 사용자 교대 점검 담당자 · 자료 가상 규정 rules-v1 · 목표 질문에 맞는 근거와 개정일 확인 · 제외 실제 점검 승인, 직원 평가, 예약·결제, 실제 사용자 인증. 주제를 검토하는 동료에게는 “찾은 원문을 보고 다음에 무엇을 할 수 있나?”라고 물어보세요.


### 팀 참고 예시 · 장비별 예시 입력


In [2]:
other_rule = {'id': 'RULE-B',
 'equipment': '장비 B',
 'text': '점검 전 고정 장치를 확인합니다.',
 'source': '가상 장비실 규정 B',
 'page': 2,
 'section': '2.2',
 'revised_on': '2026-11-01',
 'version': 'rules-v1',
 'groups': ['public', 'admin']}
print(f"{other_rule['text']} ({other_rule['section']}절)")

점검 전 고정 장치를 확인합니다. (2.2절)


“점검 전 고정 장치를 확인합니다. (2.2절)”이 됩니다. 주제를 바꿀 때도 이처럼 질문과 원문을 한 쌍으로 남기세요.


#### 팀의 첫 시연 질문을 정할 때

<mark>장비 이름이 바뀌면 정답 문장과 근거 절도 함께 바뀌어야 합니다.</mark> 질문의 단어 하나만 바꾼 짝을 준비하세요. 같은 답을 항상 출력하는 예시도 첫 질문 하나는 맞힐 수 있으므로 두 입력을 번갈아 시연합니다.

선수 개념 · 43장 RAG 프로젝트의 문제 정의와 수용 기준

## 첫 시연 계약서에 질문과 사용자의 마지막 행동 적기

규정 검색과 점검 승인 중 무엇을 먼저 개발할지 결정하는 팀 회의입니다. 두 기능 모두 쓸모 있어 보여도 이번 팀이 가진 자료로 확인할 수 있는 행동은 다릅니다.

### 가상 장비실 팀의 주제 선정 회의록

<table><thead><tr><th scope="col">후보</th><th scope="col">손에 있는 자료</th><th scope="col">시연을 끝내는 사용자 행동</th><th scope="col">결정</th></tr></thead><tbody><tr><td>점검 전 확인사항 조회</td><td>가상 규정 A·B의 본문, 절, 개정일</td><td>RULE-A의 2.1절을 열고 화면의 안내와 원문을 대조한다.</td><td>첫 시연에 포함</td></tr><tr><td>점검 승인 요청</td><td>승인자 목록·권한·결재 이력 미확보</td><td>담당자가 요청을 승인하고 이력을 다시 조회한다.</td><td>자료 확보 전 보류</td></tr></tbody></table>

‘점검을 도와준다’는 두 후보에 모두 해당합니다. 마지막 행동을 적으면 조회를 만드는 팀과 실제 업무 상태를 바꾸는 팀의 준비물이 갈립니다.

### 동료에게 건네는 첫 시연 의뢰

“질문은 ‘장비 A의 점검 전 확인 사항은?’입니다. 답변의 ‘전원 표시등’이 어느 절에서 왔는지 찾아 주세요. 찾으면 원문 첫 문장을 소리 내어 읽어 주세요.” 어떤 버튼을 누를지는 알려 주지 않습니다.

### 팀 주제로 바꾸기

문서의 이름과 사용자의 마지막 행동을 한 쌍으로 바꾸세요. 학사 규정이라면 ‘신청 가능 여부를 승인한다’와 ‘해당 학기의 신청 기한이 적힌 절을 찾는다’가 각각 어떤 자료를 요구하는지 비교해 한 쪽을 고릅니다.

<mark>사용자가 마지막에 직접 할 행동이 시연의 종료 조건입니다.</mark> 답이 화면에 뜬 뒤 원문을 찾을 수 없으면 조회 시연은 아직 끝나지 않았습니다.

87장에 전달할 것은 선택한 질문 한 문장, 공개할 수 있는 근거 위치, 도움 없이 관찰할 마지막 행동입니다.

## 팀 사례에 적용하기 · 배포 판단에서 첫 시연 조건 고르기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
new = {"previous_release": incoming,
       "scenario": {"id": "Q01", "question": "장비 A의 점검 전 확인 사항은?",
                    "source_id": rule["id"], "expected": rule["text"]},
       "acceptance": "사용자가 답을 읽고 원문 쪽과 절을 찾는다."}
print(new["scenario"]["question"], new["acceptance"])

장비 A의 점검 전 확인 사항은? 사용자가 답을 읽고 원문 쪽과 절을 찾는다.


표본의 배포 판단이 보류여도 팀은 기획을 시작할 수 있습니다. 이전 서비스의 실패 사례와 이번 서비스에서 보여 줄 규정은 따로 기록합니다.

In [ ]:
output_file = project_root / "team-project/final/chapter86.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter86.json


87장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

다음 장에는 사용자 한 사람, 그 사람이 할 질문 한 문장, 원문으로 돌아갈 단서와 제외할 업무를 함께 넘깁니다. 팀 주제가 바뀌면 이 네 가지가 어떻게 연결되는지부터 다시 읽어 보세요.
